# 🚦 Traffic Sign Recognition — PYNQ-Z2 No-Camera Demo

**Board:** PYNQ-Z2 (XC7Z020 ARM Cortex-A9)  
**Mode:** CPU-only inference — **No camera / No overlay / No VDMA needed**  
**Dataset:** GTSRB (43 traffic sign classes)  
**Model:** `gtsrb_quantized.tflite` (INT8)

---
> 🔑 **How it works:** The TFLite model runs on the ARM CPU (PS side).
> The FPGA fabric is only used for video capture. Without a camera,
> we feed **test images** directly — results are identical.

### Files needed on the board (`/home/xilinx/`):
- `gtsrb_quantized.tflite`
- This notebook (`tsr_no_camera.ipynb`)

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Install tflite-runtime if not present
try:
    import tflite_runtime
    print('✅ tflite_runtime already installed')
except ImportError:
    print('Installing tflite-runtime...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tflite-runtime'])
    print('✅ tflite_runtime installed')

import tflite_runtime.interpreter as tflite
import numpy as np
import cv2
import time
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display, HTML

print('✅ All imports successful')

## Step 2 — GTSRB Class Labels

In [ ]:
GTSRB_LABELS = {
    0:  'Speed limit (20km/h)',    1:  'Speed limit (30km/h)',    2:  'Speed limit (50km/h)',
    3:  'Speed limit (60km/h)',    4:  'Speed limit (70km/h)',    5:  'Speed limit (80km/h)',
    6:  'End of speed limit (80km/h)',                            7:  'Speed limit (100km/h)',
    8:  'Speed limit (120km/h)',   9:  'No passing',
    10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection',                  12: 'Priority road',
    13: 'Yield',                   14: 'Stop',                    15: 'No vehicles',
    16: 'Vehicles over 3.5 metric tons prohibited',               17: 'No entry',
    18: 'General caution',         19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right',                           21: 'Double curve',
    22: 'Bumpy road',              23: 'Slippery road',           24: 'Road narrows on the right',
    25: 'Road work',               26: 'Traffic signals',          27: 'Pedestrians',
    28: 'Children crossing',       29: 'Bicycles crossing',        30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed and passing limits',                    33: 'Turn right ahead',
    34: 'Turn left ahead',         35: 'Ahead only',              36: 'Go straight or right',
    37: 'Go straight or left',     38: 'Keep right',              39: 'Keep left',
    40: 'Roundabout mandatory',    41: 'End of no passing',
    42: 'End of no passing by vehicles over 3.5 metric tons'
}
print(f'✅ {len(GTSRB_LABELS)} class labels loaded')

## Step 3 — Download GTSRB Test Images
Downloads sample traffic sign images from the GTSRB dataset (public).  
Or skip this and upload your own images into `test_images/` folder.

In [ ]:
# Download a set of GTSRB test images from public GitHub mirror
# These are real test images from the German Traffic Sign Recognition Benchmark

os.makedirs('test_images', exist_ok=True)

# GTSRB test images — raw GitHub (one per common class)
# Format: (filename, url, true_class_id)
TEST_IMAGES = [
    ('stop.jpg',          'https://github.com/mohamedaminebelghith/GTSRB-Classification/raw/main/sample_images/stop.jpg',          14),
    ('speed_50.jpg',      'https://github.com/mohamedaminebelghith/GTSRB-Classification/raw/main/sample_images/speed50.jpg',       2),
    ('no_entry.jpg',      'https://github.com/mohamedaminebelghith/GTSRB-Classification/raw/main/sample_images/no_entry.jpg',      17),
    ('yield.jpg',         'https://github.com/mohamedaminebelghith/GTSRB-Classification/raw/main/sample_images/yield.jpg',         13),
    ('priority_road.jpg', 'https://github.com/mohamedaminebelghith/GTSRB-Classification/raw/main/sample_images/priority_road.jpg', 12),
]

downloaded = []
for fname, url, cls in TEST_IMAGES:
    path = f'test_images/{fname}'
    if os.path.exists(path):
        print(f'  ✅ Already exists: {fname}')
        downloaded.append((path, cls, fname))
        continue
    try:
        urllib.request.urlretrieve(url, path)
        print(f'  ✅ Downloaded: {fname}  (class {cls}: {GTSRB_LABELS[cls]})')
        downloaded.append((path, cls, fname))
    except Exception as e:
        print(f'  ⚠️  Could not download {fname}: {e}')
        # Fallback: create a synthetic test image (colored square)
        img = np.zeros((100, 100, 3), dtype=np.uint8)
        colors = {14:(255,0,0), 2:(0,100,255), 17:(200,0,0), 13:(255,200,0), 12:(255,200,0)}
        img[:] = colors.get(cls, (128,128,128))
        cv2.imwrite(path, img)
        print(f'  🔄  Created synthetic placeholder: {fname}')
        downloaded.append((path, cls, fname))

print(f'\n✅ {len(downloaded)} test images ready')

## Step 4 — Load TFLite Model

In [ ]:
MODEL_PATH = 'gtsrb_quantized.tflite'

interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
in_det  = interpreter.get_input_details()
out_det = interpreter.get_output_details()

print(f'✅ Model loaded: {MODEL_PATH}')
print(f'   Size         : {os.path.getsize(MODEL_PATH)/1024:.1f} KB')
print(f'   Input shape  : {in_det[0]["shape"]}')
print(f'   Input dtype  : {in_det[0]["dtype"]}')
print(f'   Output shape : {out_det[0]["shape"]}')

## Step 5 — Run Inference on All Test Images

In [ ]:
def infer(img_bgr):
    """Run one frame through the model. Returns (pred_class, confidence%, latency_ms)."""
    img_rgb     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (100, 100))

    if in_det[0]['dtype'] == np.float32:
        data = np.expand_dims(img_resized, 0).astype(np.float32) / 255.0
    else:
        data = np.expand_dims(img_resized, 0).astype(np.uint8)

    t0 = time.time()
    interpreter.set_tensor(in_det[0]['index'], data)
    interpreter.invoke()
    lat = (time.time() - t0) * 1000

    out  = interpreter.get_tensor(out_det[0]['index'])[0]
    pred = int(np.argmax(out))
    conf = (float(out[pred]) * 100) if out_det[0]['dtype'] == np.float32 \
           else (float(out[pred]) / 255.0 * 100)
    return pred, conf, lat

results = []
for img_path, true_cls, fname in downloaded:
    img = cv2.imread(img_path)
    if img is None:
        continue
    pred, conf, lat = infer(img)
    results.append({
        'path': img_path, 'name': fname,
        'true_cls': true_cls, 'true_label': GTSRB_LABELS.get(true_cls, '?'),
        'pred_cls':  pred,     'pred_label':  GTSRB_LABELS.get(pred,  '?'),
        'confidence': conf,    'latency_ms':  lat,
        'correct': (pred == true_cls)
    })

print(f'✅ Inference complete on {len(results)} images')

## Step 6 — Visual Results Grid

In [ ]:
n     = len(results)
cols  = min(n, 5)
rows  = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 4))
if rows == 1: axes = [axes]
if cols == 1: axes = [[ax] for ax in axes]

for i, r in enumerate(results):
    ax = axes[i // cols][i % cols]
    img_bgr = cv2.imread(r['path'])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    color  = '#2ecc71' if r['correct'] else '#e74c3c'
    status = '✅ CORRECT' if r['correct'] else '❌ WRONG'

    ax.set_title(
        f"{status}\n{r['pred_label']}\nConf: {r['confidence']:.1f}%  |  {r['latency_ms']:.1f}ms",
        fontsize=9, color=color, weight='bold'
    )
    ax.set_xlabel(f"True: {r['true_label']}", fontsize=8, color='gray')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)

# Hide empty subplots
for j in range(n, rows * cols):
    axes[j // cols][j % cols].axis('off')

fig.suptitle('PYNQ-Z2 Traffic Sign Recognition — Test Image Results',
             fontsize=13, weight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tsr_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 Results saved to tsr_results.png')

## Step 7 — Results Table

In [ ]:
df = pd.DataFrame([{
    'Image':      r['name'],
    'True Sign':  r['true_label'],
    'Predicted':  r['pred_label'],
    'Confidence': f"{r['confidence']:.1f}%",
    'Latency':    f"{r['latency_ms']:.2f} ms",
    'Result':     '✅ Correct' if r['correct'] else '❌ Wrong'
} for r in results])

display(df.style.applymap(
    lambda v: 'color: green; font-weight: bold' if '✅' in str(v) else
              ('color: red; font-weight: bold'  if '❌' in str(v) else ''),
    subset=['Result']
))

## Step 8 — Performance Summary

In [ ]:
import os

n_correct   = sum(1 for r in results if r['correct'])
avg_lat     = np.mean([r['latency_ms']  for r in results])
avg_conf    = np.mean([r['confidence']  for r in results])
accuracy    = n_correct / len(results) * 100
model_size  = os.path.getsize(MODEL_PATH) / 1024

display(HTML(f"""
<table style='border-collapse:collapse; font-family:monospace; width:500px'>
  <tr style='background:#2c3e50;color:white'>
      <th colspan=2 style='padding:10px'>📊 PYNQ-Z2 TSR Project — Summary</th></tr>
  <tr><td style='padding:8px;background:#f9f9f9'><b>Board</b></td>
      <td>PYNQ-Z2 (ARM Cortex-A9 @ 650 MHz)</td></tr>
  <tr><td style='padding:8px'><b>Mode</b></td>
      <td>CPU-only inference (no camera required)</td></tr>
  <tr><td style='padding:8px;background:#f9f9f9'><b>Model</b></td>
      <td>gtsrb_quantized.tflite (INT8, {model_size:.0f} KB)</td></tr>
  <tr><td style='padding:8px'><b>Dataset</b></td>
      <td>GTSRB — 43 traffic sign classes</td></tr>
  <tr><td style='padding:8px;background:#f9f9f9'><b>Images Tested</b></td>
      <td>{len(results)}</td></tr>
  <tr><td style='padding:8px'><b>Accuracy</b></td>
      <td style='color:{"green" if accuracy>=80 else "red"}'><b>{n_correct}/{len(results)} = {accuracy:.1f}%</b></td></tr>
  <tr><td style='padding:8px;background:#f9f9f9'><b>Avg Confidence</b></td>
      <td>{avg_conf:.2f}%</td></tr>
  <tr><td style='padding:8px'><b>Avg Latency</b></td>
      <td>{avg_lat:.2f} ms  (~{1000/avg_lat:.1f} FPS equivalent)</td></tr>
</table>
"""))

## Step 9 — Confidence Bar Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: Confidence per image
names  = [r['name'].replace('.jpg','') for r in results]
confs  = [r['confidence'] for r in results]
colors = ['#2ecc71' if r['correct'] else '#e74c3c' for r in results]
bars   = ax1.bar(names, confs, color=colors, edgecolor='white', linewidth=1.5)
ax1.axhline(85, color='orange', linestyle='--', linewidth=1.5, label='85% threshold')
ax1.set_ylim(0, 105)
ax1.set_ylabel('Confidence (%)')
ax1.set_title('Prediction Confidence per Image')
ax1.tick_params(axis='x', rotation=20)
for bar, conf in zip(bars, confs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{conf:.0f}%', ha='center', va='bottom', fontsize=9)
green_p = mpatches.Patch(color='#2ecc71', label='Correct')
red_p   = mpatches.Patch(color='#e74c3c', label='Wrong')
ax1.legend(handles=[green_p, red_p, plt.Line2D([0],[0],color='orange',linestyle='--',label='85% line')])

# Bar chart: Latency per image
lats = [r['latency_ms'] for r in results]
ax2.bar(names, lats, color='#3498db', edgecolor='white', linewidth=1.5)
ax2.axhline(np.mean(lats), color='red', linestyle='--', linewidth=1.5,
            label=f'Avg: {np.mean(lats):.1f}ms')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Inference Latency per Image')
ax2.tick_params(axis='x', rotation=20)
ax2.legend()

plt.suptitle('PYNQ-Z2 TSR Performance Analysis', weight='bold')
plt.tight_layout()
plt.savefig('tsr_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 Performance charts saved to tsr_performance.png')

---
## ✅ Demo Complete!

| Output File | Contents |
|---|---|
| `tsr_results.png` | Visual grid of images + predictions |
| `tsr_performance.png` | Confidence + latency bar charts |

> **Want live inference?** Plug a USB webcam into the PYNQ-Z2 and run:  
> `sudo python3 pynq_usb_webcam.py`

*PYNQ-Z2 TSR Project — April 2026*